# 🦷 Dental Caries Detection — Enhanced Production Pipeline
### Multi-YOLO Benchmark · Classification · Segmentation · Deployment

---

## 📋 Pipeline Overview

| Step | Section | What happens |
|------|---------|-------------|
| 0 | **Environment & GPU** | Setup, verify hardware |
| 1 | **Install & Dataset** | Roboflow download, YAML fix |
| 2 | **EDA** | Class distribution, image stats, sample grid |
| 3 | **Preprocessing** | Normalization, augmentation pipeline |
| 4 | **Detection Benchmark** | YOLOv5s/m · YOLOv8s/m · YOLO11n/s/m/l/x — 30 epochs each |
| 5 | **Segmentation Training** | YOLOv8-seg · YOLO11-seg on best architecture |
| 6 | **Winner Retrain** | 100 epochs, full augmentation, LR scheduling |
| 7 | **Evaluation** | mAP, Precision, Recall, F1, Confusion Matrix |
| 8 | **Visualization** | Boxes, masks, before/after, PR curves |
| 9 | **Export + Flask API** | ONNX export, REST API ready to deploy |

> ⚡ **Requires GPU** — Runtime → Change Runtime Type → **T4 GPU**

---
## ⚙️ Step 0 — Environment & GPU Check

In [ ]:
# ── GPU info ────────────────────────────────────────────────────────────────
!nvidia-smi
import os, sys
HOME = os.getcwd()
print(f"\n📁 Working directory: {HOME}")

---
## 📦 Step 1 — Install Dependencies & Download Dataset

In [ ]:
# ── Install all required libraries ──────────────────────────────────────────
%pip install ultralytics roboflow albumentations flask flask-cors pyngrok --quiet
%pip install scikit-learn seaborn matplotlib opencv-python-headless --quiet

# YOLOv5 (classic baseline)
!git clone -q https://github.com/ultralytics/yolov5.git
%pip install -q -r yolov5/requirements.txt

import ultralytics
ultralytics.checks()
print('\n✅ All dependencies installed')

In [ ]:
# ── Download dataset from Roboflow ──────────────────────────────────────────
import os
os.makedirs(f"{HOME}/datasets", exist_ok=True)
%cd {HOME}/datasets

from roboflow import Roboflow

# Detection dataset (YOLOv11 format = YOLOv8 compatible)
rf      = Roboflow(api_key="AThA4bwu6OviwgEwdB0v")
project = rf.workspace("maryam-sayed-ahmed").project("cavity-rs0uf-wczha")
version = project.version(1)
dataset = version.download("yolov11")

# ── Fix data.yaml paths ─────────────────────────────────────────────────────
import yaml
yaml_path = f"{dataset.location}/data.yaml"
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

cfg['train'] = f"{dataset.location}/train/images"
cfg['val']   = f"{dataset.location}/valid/images"
cfg['test']  = f"{dataset.location}/test/images"

with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f"\n✅ Dataset ready at: {dataset.location}")
print(f"   Classes: {cfg.get('names')}")

DATA_YAML = yaml_path
%cd {HOME}

---
## 🔎 Step 2 — Exploratory Data Analysis (EDA)

In [ ]:
# ── 2.1  Dataset split counts ───────────────────────────────────────────────
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
import cv2

splits = {'train': 'train', 'val': 'valid', 'test': 'test'}
split_info = {}
for name, folder in splits.items():
    imgs  = glob.glob(f"{dataset.location}/{folder}/images/*.jpg")
    imgs += glob.glob(f"{dataset.location}/{folder}/images/*.png")
    lbls  = glob.glob(f"{dataset.location}/{folder}/labels/*.txt")
    split_info[name] = {'images': len(imgs), 'labels': len(lbls)}

df_split = pd.DataFrame(split_info).T
print("📊 Dataset split summary")
print(df_split.to_string())
print(f"\n   Total images: {df_split['images'].sum()}")

In [ ]:
# ── 2.2  Class distribution across all splits ───────────────────────────────
import yaml
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
class_names = cfg.get('names', {})

all_class_ids = []
all_bbox_areas = []
img_widths, img_heights = [], []

for folder in ['train', 'valid', 'test']:
    for lbl_file in glob.glob(f"{dataset.location}/{folder}/labels/*.txt"):
        with open(lbl_file) as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    w, h   = float(parts[3]), float(parts[4])
                    all_class_ids.append(cls_id)
                    all_bbox_areas.append(w * h * 100)  # % of image

    for img_file in glob.glob(f"{dataset.location}/{folder}/images/*.jpg"):
        img = Image.open(img_file)
        img_widths.append(img.width)
        img_heights.append(img.height)

# ── Plots ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('EDA — Dental Caries Dataset', fontsize=14, fontweight='bold')

# Class distribution
unique, counts = np.unique(all_class_ids, return_counts=True)
names = [class_names.get(int(i), f'cls_{i}') if isinstance(class_names, dict)
         else class_names[int(i)] for i in unique]
axes[0].bar(names, counts, color=['#4C9BE8', '#E87B4C', '#4CE87B'][:len(unique)])
axes[0].set_title('Class Distribution (all annotations)')
axes[0].set_ylabel('Count')
for i, (n, c) in enumerate(zip(names, counts)):
    axes[0].text(i, c + 5, str(c), ha='center', fontweight='bold')

# Bounding box area
axes[1].hist(all_bbox_areas, bins=40, color='#4C9BE8', edgecolor='white', alpha=0.85)
axes[1].set_title('Bounding Box Area Distribution (% of image)')
axes[1].set_xlabel('Area (%)')
axes[1].set_ylabel('Count')

# Image resolution
axes[2].scatter(img_widths, img_heights, alpha=0.4, s=15, color='#E84C6C')
axes[2].set_title('Image Resolutions')
axes[2].set_xlabel('Width (px)')
axes[2].set_ylabel('Height (px)')

plt.tight_layout()
plt.savefig(f"{HOME}/eda_stats.png", dpi=120, bbox_inches='tight')
plt.show()
print(f"\n📐 Avg image size: {int(np.mean(img_widths))}×{int(np.mean(img_heights))} px")
print(f"   Total annotations: {len(all_class_ids)}")

In [ ]:
# ── 2.3  Sample image grid with bounding boxes ──────────────────────────────
def draw_yolo_boxes(img_path, lbl_path, class_names):
    """Draw YOLO bounding boxes on an image. Returns annotated array."""
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    colors = [(76, 155, 232), (232, 123, 76), (76, 232, 123), (232, 76, 108)]

    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:5])
                x1 = int((cx - bw/2) * w); y1 = int((cy - bh/2) * h)
                x2 = int((cx + bw/2) * w); y2 = int((cy + bh/2) * h)
                color = colors[cls_id % len(colors)]
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                label = class_names[cls_id] if isinstance(class_names, list) else class_names.get(cls_id, str(cls_id))
                cv2.putText(img, label, (x1, max(y1-6, 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    return img

train_imgs = sorted(glob.glob(f"{dataset.location}/train/images/*.jpg"))[:12]
fig, axes = plt.subplots(3, 4, figsize=(20, 14))
fig.suptitle('Sample Training Images with Ground Truth Boxes', fontsize=14, fontweight='bold')

for ax, img_path in zip(axes.flatten(), train_imgs):
    lbl_path = img_path.replace('/images/', '/labels/').replace('.jpg', '.txt')
    annotated = draw_yolo_boxes(img_path, lbl_path, class_names)
    ax.imshow(annotated)
    ax.axis('off')
    ax.set_title(os.path.basename(img_path), fontsize=8)

plt.tight_layout()
plt.savefig(f"{HOME}/sample_grid.png", dpi=100, bbox_inches='tight')
plt.show()

---
## 🔄 Step 3 — Data Preprocessing & Augmentation

In [ ]:
# ── 3.1  Visualize augmentation pipeline ────────────────────────────────────
# NOTE: YOLO handles augmentation internally via its training args.
# Here we visualize what those augmentations look like on real images.

import albumentations as A

augment_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.1),
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=30, val_shift_limit=20, p=0.5),
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),
    A.CLAHE(p=0.4),                            # Enhance local contrast (useful for X-rays)
    A.RandomGamma(gamma_limit=(80, 120), p=0.3),
    A.CoarseDropout(max_holes=4, max_height=30, max_width=30, p=0.3),
])

sample_img_path = train_imgs[0]
orig = cv2.cvtColor(cv2.imread(sample_img_path), cv2.COLOR_BGR2RGB)
orig_resized = cv2.resize(orig, (416, 416))

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
fig.suptitle('Augmentation Pipeline — Effect on Sample Image', fontsize=14, fontweight='bold')

axes[0][0].imshow(orig_resized)
axes[0][0].set_title('Original', fontweight='bold')
axes[0][0].axis('off')

for i, ax in enumerate(list(axes.flatten())[1:]):
    aug = augment_pipeline(image=orig_resized)
    ax.imshow(aug['image'])
    ax.set_title(f'Augmented #{i+1}')
    ax.axis('off')

plt.tight_layout()
plt.savefig(f"{HOME}/augmentation_preview.png", dpi=100, bbox_inches='tight')
plt.show()
print('✅ Augmentation pipeline visualized')
print('   YOLO applies equivalent augmentations during training automatically')

In [ ]:
# ── 3.2  Image normalization stats ──────────────────────────────────────────
# Compute mean and std of the training set (for reference/custom models)
pixel_vals = []
for img_path in train_imgs[:50]:   # sample 50 images for speed
    img = cv2.imread(img_path).astype(np.float32) / 255.0
    pixel_vals.append(img.reshape(-1, 3))

pixel_vals = np.concatenate(pixel_vals, axis=0)
mean = pixel_vals.mean(axis=0)
std  = pixel_vals.std(axis=0)

print('📐 Dataset Normalization Statistics (RGB channels)')
print(f'   Mean : R={mean[2]:.4f}  G={mean[1]:.4f}  B={mean[0]:.4f}')
print(f'   Std  : R={std[2]:.4f}  G={std[1]:.4f}  B={std[0]:.4f}')
print('\n   Note: YOLO normalizes to [0,1] internally — these stats are')
print('   useful if you build a custom CNN head on top of YOLO features.')

---
## 🚀 Step 4 — Multi-Model Detection Benchmark

We train **9 models** for 30 epochs each:
- `yolov5s`, `yolov5m` — classic baseline
- `yolov8s`, `yolov8m` — modern architecture
- `yolo11n`, `yolo11s`, `yolo11m`, `yolo11l`, `yolo11x` — latest YOLO11

All use identical hyperparameters for a fair comparison.
After benchmark, the winner is retrained for 100 epochs in Step 6.

In [ ]:
# ── 4.1  Define model list ───────────────────────────────────────────────────
DETECTION_MODELS = [
    # (model_weight, display_name, family)
    ("yolov5s.pt",  "YOLOv5s",  "YOLOv5"),
    ("yolov5m.pt",  "YOLOv5m",  "YOLOv5"),
    ("yolov8s.pt",  "YOLOv8s",  "YOLOv8"),
    ("yolov8m.pt",  "YOLOv8m",  "YOLOv8"),
    ("yolo11n.pt",  "YOLO11n",  "YOLO11"),
    ("yolo11s.pt",  "YOLO11s",  "YOLO11"),
    ("yolo11m.pt",  "YOLO11m",  "YOLO11"),
    ("yolo11l.pt",  "YOLO11l",  "YOLO11"),
    ("yolo11x.pt",  "YOLO11x",  "YOLO11"),
]

# ── Benchmark hyperparameters (same for ALL models) ─────────────────────────
BENCH_EPOCHS = 30
BENCH_IMGSZ  = 640
BENCH_BATCH  = 16
BENCH_PROJ   = f"{HOME}/runs/benchmark"

print(f"Models to benchmark : {len(DETECTION_MODELS)}")
print(f"Epochs each         : {BENCH_EPOCHS}")
print(f"Image size          : {BENCH_IMGSZ}")
print(f"Batch size          : {BENCH_BATCH}")

In [ ]:
# ── 4.2  Run benchmark training loop ────────────────────────────────────────
import time
from ultralytics import YOLO as UltralyticsYOLO
import subprocess

%cd {HOME}
os.makedirs(BENCH_PROJ, exist_ok=True)

benchmark_times = {}

for weight, display, family in DETECTION_MODELS:
    run_name = weight.replace('.pt', '')
    result_dir = f"{BENCH_PROJ}/{run_name}"

    # Skip if already trained
    if os.path.exists(f"{result_dir}/results.csv"):
        print(f"[SKIP] {display} — already trained")
        continue

    print(f"\n{'='*60}")
    print(f"  Training: {display} ({family})")
    print(f"{'='*60}")
    t0 = time.time()

    # YOLOv5 uses different CLI
    if family == 'YOLOv5':
        !python yolov5/train.py \
            --weights {weight} \
            --data {DATA_YAML} \
            --epochs {BENCH_EPOCHS} \
            --imgsz {BENCH_IMGSZ} \
            --batch-size {BENCH_BATCH} \
            --patience 10 \
            --optimizer SGD \
            --project {BENCH_PROJ} \
            --name {run_name} \
            --exist-ok \
            --cache
    else:
        !yolo task=detect mode=train \
            model={weight} \
            data={DATA_YAML} \
            epochs={BENCH_EPOCHS} \
            imgsz={BENCH_IMGSZ} \
            batch={BENCH_BATCH} \
            patience=10 \
            optimizer=SGD \
            lr0=0.01 \
            momentum=0.937 \
            weight_decay=0.0005 \
            mosaic=1.0 \
            fliplr=0.5 \
            hsv_s=0.7 \
            project={BENCH_PROJ} \
            name={run_name} \
            exist_ok=True

    elapsed = time.time() - t0
    benchmark_times[run_name] = elapsed
    print(f"  ⏱ Finished in {elapsed/60:.1f} min")

print("\n✅ Benchmark training complete")

In [ ]:
# ── 4.3  Parse results & build comparison table ─────────────────────────────
import pandas as pd
import glob as glob_module

results_data = []

for weight, display, family in DETECTION_MODELS:
    run_name = weight.replace('.pt', '')

    # YOLOv5 results format differs slightly
    csv_path = f"{BENCH_PROJ}/{run_name}/results.csv"
    if not os.path.exists(csv_path):
        print(f"[SKIP] {display} — results.csv not found at {csv_path}")
        continue

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    # Column names differ between YOLOv5 and v8/v11
    if family == 'YOLOv5':
        map50_col    = 'metrics/mAP_0.5'
        map5095_col  = 'metrics/mAP_0.5:0.95'
        prec_col     = 'metrics/precision'
        rec_col      = 'metrics/recall'
    else:
        map50_col    = 'metrics/mAP50(B)'
        map5095_col  = 'metrics/mAP50-95(B)'
        prec_col     = 'metrics/precision(B)'
        rec_col      = 'metrics/recall(B)'

    try:
        best = df.loc[df[map50_col].idxmax()]
        p    = float(best[prec_col])
        r    = float(best[rec_col])
        f1   = 2*p*r/(p+r+1e-8)

        # Model params (M)
        wt_path = f"{BENCH_PROJ}/{run_name}/weights/best.pt"
        params = 0
        if os.path.exists(wt_path):
            try:
                import torch
                ckpt = torch.load(wt_path, map_location='cpu')
                m = ckpt.get('model', ckpt)
                params = sum(p2.numel() for p2 in m.parameters()) / 1e6
            except: pass

        results_data.append({
            'Model'       : display,
            'Family'      : family,
            'Params (M)'  : round(params, 1),
            'mAP@50'      : round(float(best[map50_col]), 4),
            'mAP@50-95'   : round(float(best[map5095_col]), 4),
            'Precision'   : round(p, 4),
            'Recall'      : round(r, 4),
            'F1-Score'    : round(f1, 4),
            'Train Time'  : f"{benchmark_times.get(run_name, 0)/60:.1f} min"
        })
    except Exception as e:
        print(f"[ERROR] {display}: {e}")

results_df = pd.DataFrame(results_data).sort_values('mAP@50', ascending=False).reset_index(drop=True)

print('\n' + '='*75)
print('  DETECTION BENCHMARK — ALL MODELS RANKED BY mAP@50')
print('='*75)
print(results_df.to_string(index=False))
print('='*75)

In [ ]:
# ── 4.4  Visual comparison chart ────────────────────────────────────────────
if len(results_df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle('Multi-Model Benchmark — Detection Performance', fontsize=14, fontweight='bold')

    family_colors = {'YOLOv5': '#FF6B6B', 'YOLOv8': '#4ECDC4', 'YOLO11': '#4C9BE8'}
    bar_colors = [family_colors.get(f, '#999') for f in results_df['Family']]

    # mAP@50
    bars = axes[0].barh(results_df['Model'], results_df['mAP@50'],
                        color=bar_colors, edgecolor='white', height=0.7)
    axes[0].set_title('mAP@50', fontweight='bold')
    axes[0].set_xlim(0, 1)
    axes[0].invert_yaxis()
    for bar, val in zip(bars, results_df['mAP@50']):
        axes[0].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                     f'{val:.3f}', va='center', fontsize=9)

    # Precision vs Recall
    x = np.arange(len(results_df))
    w = 0.35
    axes[1].bar(x - w/2, results_df['Precision'], w, label='Precision', color='#4C9BE8')
    axes[1].bar(x + w/2, results_df['Recall'],    w, label='Recall',    color='#E87B4C')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(results_df['Model'], rotation=45, ha='right')
    axes[1].set_title('Precision vs Recall', fontweight='bold')
    axes[1].set_ylim(0, 1)
    axes[1].legend()

    # F1-Score
    bars2 = axes[2].barh(results_df['Model'], results_df['F1-Score'],
                         color=bar_colors, edgecolor='white', height=0.7)
    axes[2].set_title('F1-Score', fontweight='bold')
    axes[2].set_xlim(0, 1)
    axes[2].invert_yaxis()
    for bar, val in zip(bars2, results_df['F1-Score']):
        axes[2].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                     f'{val:.3f}', va='center', fontsize=9)

    # Legend for families
    from matplotlib.patches import Patch
    legend_els = [Patch(color=c, label=f) for f, c in family_colors.items()]
    fig.legend(handles=legend_els, loc='lower center', ncol=3, fontsize=10)

    plt.tight_layout(rect=[0, 0.07, 1, 1])
    plt.savefig(f"{HOME}/benchmark_comparison.png", dpi=130, bbox_inches='tight')
    plt.show()

In [ ]:
# ── 4.5  Pick winner ────────────────────────────────────────────────────────
best_model_name   = results_df.iloc[0]['Model'].lower().replace(' ', '').replace('yolo', 'yolo')
best_model_weight = results_df.iloc[0]['Model'].lower().replace(' ', '') + '.pt'
best_map50        = results_df.iloc[0]['mAP@50']
best_family       = results_df.iloc[0]['Family']

# Map display name back to run folder name
name_map = {d.lower(): w.replace('.pt','') for w,d,f in DETECTION_MODELS}
best_run_name = name_map.get(results_df.iloc[0]['Model'].lower(), best_model_name)

print(f"\n{'*'*55}")
print(f"  🏆 WINNER: {results_df.iloc[0]['Model']}")
print(f"  mAP@50   : {best_map50}")
print(f"  F1-Score : {results_df.iloc[0]['F1-Score']}")
print(f"  Family   : {best_family}")
print(f"{'*'*55}")
print(f"  Best weights: {BENCH_PROJ}/{best_run_name}/weights/best.pt")

---
## 🔬 Step 5 — Segmentation Training

We train **YOLOv8-seg** and **YOLO11-seg** for pixel-level cavity segmentation.
This produces segmentation masks in addition to bounding boxes.

> If your dataset only has bounding box annotations, YOLO will degrade gracefully.
> For full segmentation, re-export from Roboflow in **YOLOv8 Segmentation** format.

In [ ]:
# ── 5.1  Download segmentation-format dataset ───────────────────────────────
import os
os.makedirs(f"{HOME}/datasets_seg", exist_ok=True)
%cd {HOME}/datasets_seg

try:
    from roboflow import Roboflow
    rf = Roboflow(api_key="AThA4bwu6OviwgEwdB0v")
    project = rf.workspace("maryam-sayed-ahmed").project("cavity-rs0uf-wczha")
    version = project.version(1)
    dataset_seg = version.download("yolov8")

    # Fix paths
    import yaml
    seg_yaml = f"{dataset_seg.location}/data.yaml"
    with open(seg_yaml) as f:
        cfg_seg = yaml.safe_load(f)
    cfg_seg['train'] = f"{dataset_seg.location}/train/images"
    cfg_seg['val']   = f"{dataset_seg.location}/valid/images"
    cfg_seg['test']  = f"{dataset_seg.location}/test/images"
    with open(seg_yaml, 'w') as f:
        yaml.dump(cfg_seg, f)

    SEG_DATA_YAML = seg_yaml
    print(f"✅ Segmentation dataset ready: {dataset_seg.location}")
except Exception as e:
    print(f"⚠️  Using detection dataset for segmentation task: {e}")
    SEG_DATA_YAML = DATA_YAML

%cd {HOME}

In [ ]:
# ── 5.2  Train YOLOv8-seg ────────────────────────────────────────────────────
SEG_PROJ   = f"{HOME}/runs/segmentation"
SEG_EPOCHS = 50

print("Training YOLOv8s-seg...")
!yolo task=segment mode=train \
    model=yolov8s-seg.pt \
    data={SEG_DATA_YAML} \
    epochs={SEG_EPOCHS} \
    imgsz=640 \
    batch=16 \
    patience=15 \
    optimizer=AdamW \
    lr0=0.001 \
    lrf=0.01 \
    warmup_epochs=3 \
    mosaic=1.0 fliplr=0.5 hsv_s=0.7 \
    degrees=10 translate=0.1 scale=0.5 \
    project={SEG_PROJ} \
    name=yolov8s_seg \
    exist_ok=True \
    plots=True

print("\n✅ YOLOv8s-seg training complete")

In [ ]:
# ── 5.3  Train YOLO11s-seg ────────────────────────────────────────────────────
print("Training YOLO11s-seg...")
!yolo task=segment mode=train \
    model=yolo11s-seg.pt \
    data={SEG_DATA_YAML} \
    epochs={SEG_EPOCHS} \
    imgsz=640 \
    batch=16 \
    patience=15 \
    optimizer=AdamW \
    lr0=0.001 \
    lrf=0.01 \
    warmup_epochs=3 \
    mosaic=1.0 fliplr=0.5 hsv_s=0.7 \
    degrees=10 translate=0.1 scale=0.5 \
    project={SEG_PROJ} \
    name=yolo11s_seg \
    exist_ok=True \
    plots=True

print("\n✅ YOLO11s-seg training complete")

In [ ]:
# ── 5.4  Compare segmentation models ────────────────────────────────────────
from ultralytics import YOLO as UltralyticsYOLO

seg_results = []
for seg_name in ['yolov8s_seg', 'yolo11s_seg']:
    wt = f"{SEG_PROJ}/{seg_name}/weights/best.pt"
    if not os.path.exists(wt):
        print(f"[SKIP] {seg_name} weights not found")
        continue
    m = UltralyticsYOLO(wt)
    val = m.val(data=SEG_DATA_YAML, conf=0.25, iou=0.5, verbose=False)
    seg_results.append({
        'Model'        : seg_name,
        'Box mAP@50'   : round(val.box.map50, 4),
        'Seg mAP@50'   : round(val.seg.map50, 4) if hasattr(val, 'seg') else 'N/A',
        'Precision'    : round(float(val.box.p.mean()), 4),
        'Recall'       : round(float(val.box.r.mean()), 4),
    })

if seg_results:
    seg_df = pd.DataFrame(seg_results)
    print('\n' + '='*55)
    print('  SEGMENTATION MODEL COMPARISON')
    print('='*55)
    print(seg_df.to_string(index=False))

---
## 🏆 Step 6 — Retrain Winner: 100 Epochs (Full Optimization)

Now we take the best detection model and train it properly with:
- Full augmentation suite
- Cosine LR scheduling with warmup
- AdamW optimizer
- Early stopping (patience=20)
- All YOLO loss weights tuned

In [ ]:
# ── 6.1  Retrain best model for 100 epochs ───────────────────────────────────
%cd {HOME}

FINAL_PROJ = f"{HOME}/runs/final"
FINAL_NAME = f"{best_run_name}_final"

print(f"Retraining: {best_model_weight} for 100 epochs...")

if best_family == 'YOLOv5':
    !python yolov5/train.py \
        --weights {best_model_weight} \
        --data {DATA_YAML} \
        --epochs 100 \
        --imgsz 640 \
        --batch-size 16 \
        --patience 20 \
        --optimizer Adam \
        --label-smoothing 0.1 \
        --project {FINAL_PROJ} \
        --name {FINAL_NAME} \
        --exist-ok \
        --cache
else:
    !yolo task=detect mode=train \
        model={best_model_weight} \
        data={DATA_YAML} \
        epochs=100 \
        imgsz=640 \
        batch=16 \
        patience=20 \
        optimizer=AdamW \
        lr0=0.001 \
        lrf=0.001 \
        cos_lr=True \
        warmup_epochs=5 \
        warmup_momentum=0.8 \
        momentum=0.937 \
        weight_decay=0.0005 \
        label_smoothing=0.1 \
        box=7.5 cls=0.5 dfl=1.5 \
        hsv_h=0.015 hsv_s=0.7 hsv_v=0.4 \
        degrees=10.0 translate=0.1 scale=0.5 \
        shear=2.0 perspective=0.0 \
        flipud=0.1 fliplr=0.5 \
        bgr=0.0 mosaic=1.0 \
        mixup=0.1 copy_paste=0.1 erasing=0.4 \
        plots=True save=True save_period=10 \
        project={FINAL_PROJ} \
        name={FINAL_NAME} \
        exist_ok=True

FINAL_BEST = f"{FINAL_PROJ}/{FINAL_NAME}/weights/best.pt"
print(f"\n✅ Final model saved: {FINAL_BEST}")

---
## 📈 Step 7 — Comprehensive Evaluation

In [ ]:
# ── 7.1  Training curves ─────────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

csv_path = f"{FINAL_PROJ}/{FINAL_NAME}/results.csv"
df_final = pd.read_csv(csv_path)
df_final.columns = df_final.columns.str.strip()

fig, axes = plt.subplots(2, 3, figsize=(20, 11))
fig.suptitle(f'Final Training: {best_run_name} — 100 Epochs', fontsize=14, fontweight='bold')

def try_plot(ax, col_train, col_val, title):
    plotted = False
    if col_train in df_final.columns:
        ax.plot(df_final[col_train], label='Train', lw=2, color='#4C9BE8')
        plotted = True
    if col_val and col_val in df_final.columns:
        ax.plot(df_final[col_val], label='Val', lw=2, color='#E87B4C', linestyle='--')
        plotted = True
    if plotted:
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(alpha=0.3)
    else:
        ax.set_title(f'{title} (N/A)')
        ax.axis('off')

pairs = [
    ('train/box_loss',          'val/box_loss',          'Box Loss'),
    ('train/cls_loss',          'val/cls_loss',          'Classification Loss'),
    ('train/dfl_loss',          'val/dfl_loss',          'DFL Loss'),
    ('metrics/precision(B)',    None,                    'Precision'),
    ('metrics/recall(B)',       None,                    'Recall'),
    ('metrics/mAP50(B)',        None,                    'mAP@50'),
]
for ax, (tc, vc, title) in zip(axes.flatten(), pairs):
    try_plot(ax, tc, vc, title)

plt.tight_layout()
plt.savefig(f"{HOME}/final_training_curves.png", dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.2  Validation metrics — per class ─────────────────────────────────────
from ultralytics import YOLO as UltralyticsYOLO

final_model = UltralyticsYOLO(FINAL_BEST)
val_result  = final_model.val(data=DATA_YAML, conf=0.25, iou=0.6, verbose=False)

class_list = cfg.get('names', {})
if isinstance(class_list, dict):
    class_list = [class_list[k] for k in sorted(class_list.keys())]

print('\n' + '='*55)
print('  FINAL MODEL — PER CLASS METRICS')
print('='*55)

class_metrics = []
for i, cls_name in enumerate(class_list):
    try:
        p  = float(val_result.box.p[i])
        r  = float(val_result.box.r[i])
        ap = float(val_result.box.ap[i])
        f1 = 2*p*r / (p+r+1e-8)
        class_metrics.append({'Class': cls_name, 'Precision': p, 'Recall': r, 'F1': f1, 'mAP@50': ap})
        print(f'  {cls_name.upper():<20} P={p:.3f}  R={r:.3f}  F1={f1:.3f}  mAP@50={ap:.3f}')
    except IndexError:
        pass

print('─'*55)
print(f'  OVERALL mAP@50    : {val_result.box.map50:.4f}')
print(f'  OVERALL mAP@50-95 : {val_result.box.map:.4f}')
print('='*55)

In [ ]:
# ── 7.3  Confusion matrix heatmap ────────────────────────────────────────────
cm_path = f"{FINAL_PROJ}/{FINAL_NAME}/confusion_matrix_normalized.png"
if not os.path.exists(cm_path):
    cm_path = f"{FINAL_PROJ}/{FINAL_NAME}/confusion_matrix.png"

if os.path.exists(cm_path):
    img = plt.imread(cm_path)
    plt.figure(figsize=(8, 7))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confusion Matrix (Normalized)', fontweight='bold', fontsize=13)
    plt.tight_layout()
    plt.savefig(f"{HOME}/confusion_matrix.png", dpi=130, bbox_inches='tight')
    plt.show()
else:
    print("⚠️  Confusion matrix image not found — check YOLO run directory")

In [ ]:
# ── 7.4  PR Curve & F1 Curve ─────────────────────────────────────────────────
run_dir = f"{FINAL_PROJ}/{FINAL_NAME}"

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Precision-Recall & F1-Confidence Curves', fontsize=13, fontweight='bold')

for ax, (fname, title) in zip(axes, [('PR_curve.png', 'PR Curve'), ('F1_curve.png', 'F1-Confidence Curve')]):
    path = f"{run_dir}/{fname}"
    if os.path.exists(path):
        ax.imshow(plt.imread(path))
        ax.set_title(title, fontweight='bold')
    else:
        ax.set_title(f'{title} (not available)')
    ax.axis('off')

plt.tight_layout()
plt.savefig(f"{HOME}/pr_f1_curves.png", dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.5  Final benchmark summary table (detection + segmentation) ─────────────
print('\n' + '='*80)
print('  COMPLETE MODEL COMPARISON TABLE')
print('='*80)
print(results_df.to_string(index=False))

if 'seg_df' in dir() and len(seg_results) > 0:
    print('\n  SEGMENTATION MODELS')
    print('─'*55)
    print(seg_df.to_string(index=False))

print('\n  FINAL (100 EPOCH) WINNER')
print('─'*55)
print(f'  Model     : {best_run_name}')
print(f'  mAP@50    : {val_result.box.map50:.4f}')
print(f'  mAP@50-95 : {val_result.box.map:.4f}')
print('='*80)

---
## 🖼️ Step 8 — Visualization: Bounding Boxes · Masks · Before/After

In [ ]:
# ── 8.1  Run predictions on test set ────────────────────────────────────────
TEST_IMG_DIR = cfg.get('test', f"{dataset.location}/test/images")

!yolo task=detect mode=predict \
    model={FINAL_BEST} \
    source={TEST_IMG_DIR} \
    conf=0.4 iou=0.5 \
    save=True \
    project={HOME}/runs/final \
    name=predictions \
    exist_ok=True

print("✅ Predictions saved")

In [ ]:
# ── 8.2  Prediction grid — 6 test images ────────────────────────────────────
import glob as glob_module
from IPython.display import Image as IPyImg, display

pred_imgs  = sorted(glob_module.glob(f"{HOME}/runs/final/predictions/*.jpg"))
orig_imgs  = sorted(glob_module.glob(f"{TEST_IMG_DIR}/*.jpg"))

n = min(6, len(pred_imgs), len(orig_imgs))
if n > 0:
    fig, axes = plt.subplots(n, 2, figsize=(16, n * 5))
    fig.suptitle('Before/After Predictions on Test Set', fontsize=14, fontweight='bold')

    if n == 1:
        axes = [axes]

    for i in range(n):
        orig_name = os.path.basename(pred_imgs[i])
        orig_path = os.path.join(TEST_IMG_DIR, orig_name)

        orig = cv2.cvtColor(cv2.imread(orig_path), cv2.COLOR_BGR2RGB) if os.path.exists(orig_path) \
               else np.zeros((640,640,3), dtype=np.uint8)
        pred = cv2.cvtColor(cv2.imread(pred_imgs[i]), cv2.COLOR_BGR2RGB)

        axes[i][0].imshow(orig)
        axes[i][0].set_title(f'Original — {orig_name}', fontsize=10)
        axes[i][0].axis('off')

        axes[i][1].imshow(pred)
        axes[i][1].set_title(f'Predicted — {orig_name}', fontsize=10)
        axes[i][1].axis('off')

    plt.tight_layout()
    plt.savefig(f"{HOME}/before_after_predictions.png", dpi=120, bbox_inches='tight')
    plt.show()
else:
    print("⚠️  No prediction images found — ensure test set is not empty")

In [ ]:
# ── 8.3  Segmentation mask visualization ────────────────────────────────────
best_seg_weight = f"{SEG_PROJ}/yolo11s_seg/weights/best.pt"
if not os.path.exists(best_seg_weight):
    best_seg_weight = f"{SEG_PROJ}/yolov8s_seg/weights/best.pt"

if os.path.exists(best_seg_weight) and len(orig_imgs) > 0:
    seg_model = UltralyticsYOLO(best_seg_weight)
    sample_imgs = orig_imgs[:4]

    fig, axes = plt.subplots(2, 4, figsize=(22, 11))
    fig.suptitle('Segmentation Masks — Cavity Region Detection', fontsize=14, fontweight='bold')

    for i, img_path in enumerate(sample_imgs):
        results = seg_model.predict(img_path, conf=0.35, verbose=False)
        orig_img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

        axes[0][i].imshow(orig_img)
        axes[0][i].set_title('Original', fontsize=10)
        axes[0][i].axis('off')

        seg_img = results[0].plot()
        seg_img = cv2.cvtColor(seg_img, cv2.COLOR_BGR2RGB)
        axes[1][i].imshow(seg_img)
        n_det = len(results[0].boxes) if results[0].boxes else 0
        axes[1][i].set_title(f'Segmented — {n_det} detection(s)', fontsize=10)
        axes[1][i].axis('off')

    plt.tight_layout()
    plt.savefig(f"{HOME}/segmentation_masks.png", dpi=120, bbox_inches='tight')
    plt.show()
else:
    print("ℹ️  Segmentation weights not found — showing detection predictions only")

In [ ]:
# ── 8.4  Confidence score distribution ──────────────────────────────────────
all_confs = []
for img_path in orig_imgs[:30]:
    results = final_model.predict(img_path, conf=0.1, verbose=False)
    if results[0].boxes is not None and len(results[0].boxes) > 0:
        all_confs.extend(results[0].boxes.conf.cpu().numpy().tolist())

if all_confs:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Prediction Confidence Analysis', fontsize=13, fontweight='bold')

    axes[0].hist(all_confs, bins=30, color='#4C9BE8', edgecolor='white', alpha=0.85)
    axes[0].axvline(np.mean(all_confs), color='#E84C6C', lw=2, label=f'Mean = {np.mean(all_confs):.3f}')
    axes[0].axvline(0.4, color='orange', lw=2, linestyle='--', label='Threshold = 0.4')
    axes[0].set_title('Confidence Score Distribution')
    axes[0].set_xlabel('Confidence')
    axes[0].set_ylabel('Count')
    axes[0].legend()

    axes[1].boxplot(all_confs, vert=True, patch_artist=True,
                    boxprops=dict(facecolor='#4C9BE8', alpha=0.7))
    axes[1].set_title('Confidence Box Plot')
    axes[1].set_ylabel('Confidence Score')
    axes[1].set_xticks([1])
    axes[1].set_xticklabels(['Predictions'])

    plt.tight_layout()
    plt.savefig(f"{HOME}/confidence_distribution.png", dpi=120, bbox_inches='tight')
    plt.show()
    print(f"   Mean confidence : {np.mean(all_confs):.3f}")
    print(f"   High conf (>0.7): {sum(c > 0.7 for c in all_confs)} / {len(all_confs)} ({100*sum(c>0.7 for c in all_confs)/len(all_confs):.1f}%)")

---
## 🚀 Step 9 — Export Best Model + Flask Deployment API

We export the best model to **ONNX** (universal format) and build a complete **Flask REST API** for deployment.

In [ ]:
# ── 9.1  Export to ONNX ──────────────────────────────────────────────────────
model_for_export = UltralyticsYOLO(FINAL_BEST)
onnx_path = model_for_export.export(format='onnx', imgsz=640, simplify=True, opset=12)

# Also copy .pt to models/ dir
import shutil
os.makedirs(f"{HOME}/models", exist_ok=True)
shutil.copy(FINAL_BEST, f"{HOME}/models/best_detection.pt")
if onnx_path and os.path.exists(str(onnx_path)):
    shutil.copy(str(onnx_path), f"{HOME}/models/best_detection.onnx")

print(f"\n✅ Models exported to {HOME}/models/")
print(f"   PyTorch : {HOME}/models/best_detection.pt")
print(f"   ONNX    : {HOME}/models/best_detection.onnx")

In [ ]:
# ── 9.2  Write Flask REST API ────────────────────────────────────────────────
flask_app_code = '''
"""
Flask REST API — Dental Caries Detection
========================================
Endpoints:
  POST /predict          — detect cavities in uploaded image
  POST /predict_segment  — detect + segment cavities
  GET  /health           — health check
  GET  /model_info       — model metadata

Usage:
  python app.py
  curl -X POST -F "image=@tooth.jpg" http://localhost:5000/predict
"""

import os, io, base64, time
from pathlib import Path
from flask import Flask, request, jsonify
from flask_cors import CORS
from PIL import Image
import numpy as np
import cv2
from ultralytics import YOLO

# ── App config ────────────────────────────────────────────────────────────────
app = Flask(__name__)
CORS(app)

MODEL_PATH  = os.getenv('MODEL_PATH', 'models/best_detection.pt')
CONF_THRESH = float(os.getenv('CONF_THRESH', 0.4))
IOU_THRESH  = float(os.getenv('IOU_THRESH',  0.5))
MAX_SIZE    = 5 * 1024 * 1024  # 5 MB max upload

# ── Load model once at startup ────────────────────────────────────────────────
print(f"Loading model: {MODEL_PATH}")
model = YOLO(MODEL_PATH)
print("Model loaded ✅")


def preprocess_image(file_bytes: bytes) -> np.ndarray:
    """Decode uploaded image bytes to numpy array."""
    arr = np.frombuffer(file_bytes, np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError("Cannot decode image")
    return img


def image_to_b64(img_rgb: np.ndarray) -> str:
    """Encode numpy image to base64 JPEG string."""
    _, buf = cv2.imencode('.jpg', cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR),
                          [cv2.IMWRITE_JPEG_QUALITY, 85])
    return base64.b64encode(buf).decode('utf-8')


# ── Health check ──────────────────────────────────────────────────────────────
@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'model': MODEL_PATH})


# ── Model info ────────────────────────────────────────────────────────────────
@app.route('/model_info', methods=['GET'])
def model_info():
    return jsonify({
        'model_path' : MODEL_PATH,
        'conf_thresh': CONF_THRESH,
        'iou_thresh' : IOU_THRESH,
        'task'       : 'dental caries detection',
        'classes'    : model.names,
        'input_size' : 640,
    })


# ── Main predict endpoint ─────────────────────────────────────────────────────
@app.route('/predict', methods=['POST'])
def predict():
    """Detect cavities and return JSON with bounding boxes + annotated image."""
    if 'image' not in request.files:
        return jsonify({'error': 'No image field in request'}), 400

    file = request.files['image']
    if file.filename == '':
        return jsonify({'error': 'Empty filename'}), 400

    file_bytes = file.read()
    if len(file_bytes) > MAX_SIZE:
        return jsonify({'error': f'File too large (max {MAX_SIZE//1024//1024} MB)'}), 413

    try:
        img_bgr  = preprocess_image(file_bytes)
        t0       = time.time()
        results  = model.predict(img_bgr, conf=CONF_THRESH, iou=IOU_THRESH, verbose=False)
        inf_ms   = round((time.time() - t0) * 1000, 1)

        detections = []
        for box in results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            conf   = float(box.conf[0])
            cls_id = int(box.cls[0])
            detections.append({
                'class_id'  : cls_id,
                'class_name': model.names[cls_id],
                'confidence': round(conf, 4),
                'bbox'      : [round(x1), round(y1), round(x2), round(y2)],
            })

        annotated = cv2.cvtColor(results[0].plot(), cv2.COLOR_BGR2RGB)

        return jsonify({
            'status'         : 'success',
            'inference_ms'   : inf_ms,
            'num_detections' : len(detections),
            'detections'     : detections,
            'annotated_image': image_to_b64(annotated),   # base64 JPEG
        })

    except Exception as e:
        return jsonify({'error': str(e)}), 500


# ── Segment endpoint ──────────────────────────────────────────────────────────
@app.route('/predict_segment', methods=['POST'])
def predict_segment():
    """Run segmentation model (if available) and return mask overlay."""
    seg_path = os.getenv('SEG_MODEL_PATH', 'models/best_segmentation.pt')
    if not os.path.exists(seg_path):
        return jsonify({'error': 'Segmentation model not found. Train in Step 5 first.'}), 404

    seg_model = YOLO(seg_path)

    if 'image' not in request.files:
        return jsonify({'error': 'No image field'}), 400

    try:
        img_bgr = preprocess_image(request.files['image'].read())
        results = seg_model.predict(img_bgr, conf=CONF_THRESH, iou=IOU_THRESH, verbose=False)
        annotated = cv2.cvtColor(results[0].plot(), cv2.COLOR_BGR2RGB)
        return jsonify({
            'status'         : 'success',
            'num_detections' : len(results[0].boxes) if results[0].boxes else 0,
            'annotated_image': image_to_b64(annotated),
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 500


if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False)
'''

flask_dir = f"{HOME}/flask_app"
os.makedirs(flask_dir, exist_ok=True)
with open(f"{flask_dir}/app.py", 'w') as f:
    f.write(flask_app_code)
print(f"✅ Flask app written to {flask_dir}/app.py")

In [ ]:
# ── 9.3  Write requirements.txt ──────────────────────────────────────────────
reqs = """ultralytics>=8.3.0
flask>=3.0.0
flask-cors>=4.0.0
opencv-python-headless>=4.9
numpy>=1.24
pillow>=10.0
"""
with open(f"{flask_dir}/requirements.txt", 'w') as f:
    f.write(reqs)

# ── 9.4  Dockerfile ──────────────────────────────────────────────────────────
dockerfile = """FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
COPY models/ models/
EXPOSE 5000
ENV MODEL_PATH=models/best_detection.pt
ENV CONF_THRESH=0.4
CMD ["python", "app.py"]
"""
with open(f"{flask_dir}/Dockerfile", 'w') as f:
    f.write(dockerfile)

print("✅ requirements.txt and Dockerfile written")
print(f"\n📁 Flask deployment files:")
for fn in os.listdir(flask_dir):
    print(f"   {flask_dir}/{fn}")

In [ ]:
# ── 9.5  Start API with ngrok tunnel (Colab-friendly public URL) ─────────────
from pyngrok import ngrok
import subprocess, time, requests

# Copy model to flask_app/models
os.makedirs(f"{flask_dir}/models", exist_ok=True)
shutil.copy(FINAL_BEST, f"{flask_dir}/models/best_detection.pt")

# Start Flask in background
flask_proc = subprocess.Popen(
    ['python', f"{flask_dir}/app.py"],
    env={**os.environ, 'MODEL_PATH': f'{flask_dir}/models/best_detection.pt'}
)
time.sleep(5)  # Wait for startup

# Open ngrok tunnel
try:
    public_url = ngrok.connect(5000)
    print(f"\n🌐 API is LIVE at: {public_url}")
    print(f"   Health check : {public_url}/health")
    print(f"   Model info   : {public_url}/model_info")
    print(f"\n   To test:")
    print(f'   curl -X POST -F "image=@your_tooth.jpg" {public_url}/predict')
except Exception as e:
    print(f"⚠️  ngrok failed: {e}")
    print("   Flask is still running at http://localhost:5000")

---
## 🏁 Final Summary

```
╔══════════════════════════════════════════════════════════════╗
║   DENTAL CARIES DETECTION — ENHANCED PIPELINE COMPLETE      ║
╠══════════════════════════════════════════════════════════════╣
║  Step 0 : GPU verification                                   ║
║  Step 1 : Dataset download from Roboflow                     ║
║  Step 2 : EDA — class distribution, sizes, sample grid       ║
║  Step 3 : Preprocessing — normalization + augmentation viz   ║
║  Step 4 : Benchmark — 9 models × 30 epochs (YOLOv5/v8/v11)  ║
║  Step 5 : Segmentation — YOLOv8-seg + YOLO11-seg             ║
║  Step 6 : Retrain winner — 100 epochs + full augmentation    ║
║  Step 7 : Evaluation — mAP, F1, PR curves, confusion matrix  ║
║  Step 8 : Visualization — boxes, masks, before/after         ║
║  Step 9 : Export ONNX + Flask REST API + Dockerfile          ║
╠══════════════════════════════════════════════════════════════╣
║  Models benchmarked:                                         ║
║    Detection  : YOLOv5s/m · YOLOv8s/m · YOLO11n/s/m/l/x     ║
║    Segmentation: YOLOv8s-seg · YOLO11s-seg                   ║
╚══════════════════════════════════════════════════════════════╝
```